In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_emp = spark.read.format('csv').option('header', True).load('/Volumes/workspace/default/datasets/emp_data_ABC/emp_ABC_update_4.csv')
df_emp.display()

In [0]:
%sql
select * from workspace.default.emp_abc_type_2

In [0]:
df_emp.createOrReplaceTempView('emp_view')

In [0]:
%sql
select * from emp_view

In [0]:
%sql
MERGE into emp_abc
using emp_view
on emp_abc.emp_id = emp_view.emp_id
when MATCHED AND emp_view.emp_sal != emp_abc.emp_sal 
THEN UPDATE
SET
emp_abc.emp_sal=emp_view.emp_sal,
emp_abc.year = emp_view.year,
emp_abc.lst_updt_ts = current_timestamp()
when NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,create_ts,lst_updt_ts)
VALUES(emp_view.emp_id,emp_view.emp_name,emp_view.emp_sal,emp_view.year,current_timestamp(),current_timestamp())

In [0]:
%sql
MERGE INTO workspace.default.emp_abc_type_2 tgt
USING emp_view src
on tgt.emp_id =  src.emp_id AND tgt.cur_rec_ind = 'Y'
WHEN MATCHED AND tgt.emp_sal != src.emp_sal
THEN UPDATE 
SET tgt.cur_rec_ind = 'N',
    tgt.lst_updt_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,cur_rec_ind,create_ts,lst_updt_ts)
VALUES(src.emp_id,src.emp_name,src.emp_sal,src.year,'Y',current_timestamp(),current_timestamp())

In [0]:
# cur_rec_ind / is_active/ active_flag
# start_ts/create_ts/insert_ts
# end_ts/lst_updt_ts

In [0]:
%sql
select * from emp_abc_type_2 where emp_id in (123,124)

In [0]:
%sql
select * from emp_abc_type_2

In [0]:
%sql
MERGE INTO workspace.default.emp_abc_type_2 tgt
USING emp_view src
on tgt.emp_id =  src.emp_id AND tgt.cur_rec_ind = 'Y'

WHEN NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,cur_rec_ind,create_ts,lst_updt_ts)
VALUES(src.emp_id,src.emp_name,src.emp_sal,src.year,'Y',current_timestamp(),current_timestamp())

In [0]:
md5( 45830)  md5(45830)

In [0]:
%sql
MERGE INTO workspace.default.emp_abc_type_2 tgt
USING emp_view src
on md5(concat_ws('_',tgt.emp_id)) =  md5(concat_ws('_',src.emp_id)) AND tgt.cur_rec_ind = 'Y'
WHEN MATCHED AND md5(concat_ws('_',tgt.emp_sal)) != md5(concat_ws('_',src.emp_sal))
THEN UPDATE 
SET tgt.cur_rec_ind = 'N',
    tgt.lst_updt_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,cur_rec_ind,create_ts,lst_updt_ts)
VALUES(src.emp_id,src.emp_name,src.emp_sal,src.year,'Y',current_timestamp(),current_timestamp())

In [0]:
%sql
MERGE INTO workspace.default.emp_abc_type_2 tgt
USING emp_view src
on md5(concat_ws('_',tgt.emp_id)) =  md5(concat_ws('_',src.emp_id)) AND tgt.cur_rec_ind = 'Y'

WHEN NOT MATCHED THEN
INSERT (emp_id,emp_name,emp_sal,year,cur_rec_ind,create_ts,lst_updt_ts)
VALUES(src.emp_id,src.emp_name,src.emp_sal,src.year,'Y',current_timestamp(),current_timestamp())

In [0]:
%sql
select * from emp_abc_type_2

In [0]:
%sql
select *, md5(concat_ws('_',emp_sal)) as sal from emp_view

In [0]:
%sql
select * , md5(concat_ws('_',emp_sal)) from emp_abc_type_2 where cur_rec_ind='Y' and emp_id in (123,124,127,128)

In [0]:
condition 1 - insert in a batch (batch size -1000) - 1 part file contains 1000 records
condition 2 - insert individually - 1000 part files got created -- it is causing an issue

In [0]:
compaction ---combine multiple small files into a large file(1000 rec)

## what are the optimization methods you have used in your project

In [0]:
%sql
OPTIMIZE  emp_abc_type_2